In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

#####1. Create your bronze layer tables ingesting from the landing zone

In [0]:
@dlt.table(
    name = "customers_raw",
    comment = "table to be created in bronze layer"
)
def get_customers_raw():
    df = (
        spark
        .readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("pathGlobFilter","*.csv")
        .option("header", "True")
        .option("cloudFiles.schemaLocation", "/Volumes/dbcertification/scholarsnest/landing_zone/customers/schema/")
        .option("cloudFiles.inferColumnTypes","True")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load("/Volumes/dbcertification/scholarsnest/landing_zone/customers/")
        .withColumn("load_time",current_timestamp())
        )
    return df


In [0]:
@dlt.table(
    name = "invoices_raw",
    comment = "table to be created in bronze layer"
)
def get_invoices_raw():
    df = (
        spark
        .readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("pathGlobFilter","*.csv")
        .option("header", "True")
        .option("cloudFiles.schemaLocation", "/Volumes/dbcertification/scholarsnest/landing_zone/invoices/schema/")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes","True")
        .load("/Volumes/dbcertification/scholarsnest/landing_zone/invoices/")
        .withColumn("load_time",current_timestamp())
        )
    return df


#####2. Create your silver layer tables reading incremental data from bronze layer

In [0]:
@dlt.table(
    name = "customers_cleaned",
    comment = "table to be created in silver layer"
)
@dlt.expect_or_drop("valid_customers","customer_id is not null")
def get_customers_cleaned():
    df = (
            spark
            .readStream
            .format("delta")
            .table("LIVE.customers_raw")
            .selectExpr("CustomerID as customer_id","CustomerName as customer_name","load_time as load_time")
    )
    return df

In [0]:
@dlt.table(
    name = "invoices_cleaned",
    partition_cols = ["invoice_year","country"],
    comment = "partitioned table to be created in silver layer"
)
@dlt.expect_or_drop("valid_invoice_and_qty","invoice_no is not null and quantity > 0")
def get_invoices_cleaned():
    df = (spark
          .readStream
          .format("delta")
          .table("LIVE.invoices_raw")
          .selectExpr("InvoiceNo as invoice_no", 
                      "StockCode as stock_code", 
                      "Description as description",
                      "Quantity as quantity", 
                      "to_date(InvoiceDate, 'd-M-y H.m') as invoice_date", 
                      "UnitPrice as unit_price", 
                      "CustomerID as customer_id", 
                      "Country as country",
                      "year(to_date(InvoiceDate, 'd-M-y H.m')) as invoice_year", 
                      "month(to_date(InvoiceDate, 'd-M-y H.m')) as invoice_month", 
                      "load_time")
    )
    return df

#####3. Build your SCD Type 2 dimensions using CDC from silver layer

In [0]:
dlt.create_streaming_table("customers") #The empty streaming table

dlt.apply_changes(
    target = "customers",
    source = "customers_cleaned",
    keys = ["customer_id"],
    sequence_by = col("load_time"), #for handling out-of-order data
    stored_as_scd_type = 2
)

#####4. Merge into your fact table using CDC from the silver layer

In [0]:
dlt.create_streaming_table("invoices",cluster_by = ["invoice_year","country"])

dlt.apply_changes(
    target = "invoices",
    source = "invoices_cleaned",
    keys = ["invoice_no", "stock_code", "invoice_date"],
    sequence_by = col("load_time")
)

#####5. Materialize your gold layer summary using silver layer fact

In [0]:
@dlt.table(
    name = "daily_sales_uk_2022"
)
def compute_daily_sales_uk_2022():
    return(
        spark.read
        .format("delta")
        .table("LIVE.invoices")
        . where("invoice_year = 2022 AND country = 'United Kingdom'")
        .groupBy("country","invoice_year","invoice_month","invoice_date")
        .agg(expr("round(sum(quantity*unit_price),2)").alias("total_sales"))
    )